In [10]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST
from tqdm.auto import tqdm
from IPython.display import HTML
from sklearn.manifold import TSNE
# from celluloid import Camera
from IPython.display import HTML

import os


### Seed Everything ###
torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)
random.seed(0)

tensor_transforms = transforms.Compose(
    [
        transforms.Resize((32,32)),
        transforms.ToTensor()
    ]
)
data_root = os.path.expanduser("~/data/mnist/")
os.makedirs(data_root, exist_ok=True)

train_set = MNIST(data_root, train=True, download=True, transform=tensor_transforms)
test_set = MNIST(data_root, train=False, download=True, transform=tensor_transforms)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [15]:
class VectorQuantizer(nn.Module):
    def __init__(self, codebook_dim=1024, latent_dim=8) -> None:
        super().__init__()

        self.codebook_dim = codebook_dim
        self.latent_dim = latent_dim

        self.codebook = nn.Embedding(codebook_dim, latent_dim)
        self.codebook.weight.data.uniform_(-1 / codebook_dim, 1 / codebook_dim)

    def forward(self, x):
        # x : (B, L, H, W) -> (B*H*W, L)

        batch_size = x.shape[0]
        H, W = x.shape[2], x.shape[3]
        x = x.permute(0, 2, 3, 1).contiguous().view(-1, self.latent_dim)


        L2 = torch.sum(x**2, dim=1, keepdim=True) #(B*H*W, 1)
        C2 = torch.sum(self.codebook.weight**2, dim=1).unsqueeze(0) #(1, K)
        CL = x@self.codebook.weight.t() #(B*H*W, K)
        
        distances = L2 - 2*CL + C2 #(B*H*W, K) broadcasting L2 & C2

        indices = torch.argmin(distances, dim=1) #(B*H*W,)

        encodings = torch.zeros(indices.shape[0], self.codebook_dim, device=x.device)
        encodings.scatter_(1, indices.unsqueeze(1), 1)
        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        z_q = self.codebook(indices)                              # (B*H*W, L)
        z_q = z_q.view(batch_size, H, W, self.latent_dim)         # (B, H, W, L)
        z_q = z_q.permute(0, 3, 1, 2).contiguous()

        return z_q, indices, perplexity


In [16]:
class ConvolutionalVectorQuantizedVAE(nn.Module):
    def __init__(self, in_channels=1, latent_dim=4, codebook_size=512):
        super().__init__()

        self.bottleneck = latent_dim
        self.in_channels = in_channels 
        self.codebook_size = codebook_size

        # result in a (B, C, H, W) of Z_e the output channels should match the latent dimension as both capture the same essence 
        
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=5, stride=2, padding=1, bias=False), 
            nn.BatchNorm2d(8),
            nn.ReLU(), 

            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=2, padding=1, bias=False), 
            nn.BatchNorm2d(16),
            nn.ReLU(), 

            nn.Conv2d(in_channels=16, out_channels=self.bottleneck, kernel_size=3, stride=2, padding=1, bias=False), 
            nn.BatchNorm2d(self.bottleneck),
            nn.ReLU(),
        )

        # quantize Z_e into the following Z_q of same dimensions i.e. (B, C, H, W)
        self.vq = VectorQuantizer(codebook_size, latent_dim)

        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(in_channels=self.bottleneck, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(), 
            
            nn.ConvTranspose2d(in_channels=16, out_channels=8, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            
            nn.ConvTranspose2d(in_channels=8, out_channels=in_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )
    

    def forward_enc(self, x):
        conv_enc = self.encoder_conv(x)
        return conv_enc

    def quantize(self, z):
        codes, indices = self.vq(z)

        codebook_loss = torch.mean((codes - z.detach())**2)
        commitment_loss = torch.mean((codes.detach() - z)**2)

        codes = z + (codes - z).detach()
        
        return codes, codebook_loss, commitment_loss

    def forward_dec(self, x):
        codes, codebook_loss, commitment_loss = self.quantize(x)

        conv_dec = self.decoder_conv(codes)
        
        return codes, conv_dec, codebook_loss, commitment_loss
        
    def forward(self, x):
        latents = self.forward_enc(x)
        quantized_latents, decoded, codebook_loss, commitment_loss = self.forward_dec(latents)
        return latents, quantized_latents, decoded, codebook_loss, commitment_loss 

In [ ]:
data_variance = torch.var(train_set.data/255.)

def train(model,train_set, test_set, batch_size, training_iterations, evaluation_iterations):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    trainloader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=8)
    testloader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=8)

    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    train_losses = []
    evaluation_losses = []

    pbar  = tqdm(total=training_iterations, position=0, leave=True)

    train = True
    step = 0

    while train:
        for images, labels in trainloader:
            images = images.to(device)
            optimizer.zero_grad()
            latents, quantized_latents, decoded, codebook_loss, commitment_loss = model(images)
            recon_loss = torch.mean((decoded - images)**2)/data_variance
            loss = recon_loss + codebook_loss + commitment_loss
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            step += 1
            pbar.update(1)

            if step % evaluation_iterations == 0:
                model.eval()
                with torch.no_grad():
                    for images, labels in testloader:
                        images = images.to(device)
                        latents, quantized_latents, decoded, codebook_loss, commitment_loss = model(images)
                        recon_loss = torch.mean((decoded - images)**2)/data_variance
                        loss = recon_loss + codebook_loss + commitment_loss
                        evaluation_losses.append(loss.item())
                model.train()

            if step >= training_iterations:
                train = False
                break

    return model, train_losses, evaluation_losses


In [14]:
conv_vqvae = ConvolutionalVectorQuantizedVAE()
(conv_vqvae, train_losses, 
 evaluation_losses) = train(  conv_vqvae,                                                       
                                                        train_set=train_set,
                                                       test_set=test_set,
                                                       batch_size=64,
                                                       training_iterations=25000,
                                                       evaluation_iterations=250,
                                                       )

  0%|          | 0/25000 [00:00<?, ?it/s]

KeyboardInterrupt: 